In [13]:
# Guardamos nuestra API Key personal extraida para RAWG
api_key = os.getenv("OPENAI_API_KEY")
api_key

'74c8e14f7976452ebdbdc1fcf6b714c9'

In [9]:
# Importamos librerias
import pandas as pd
import os
import urllib.request, json, csv
import numpy as np
from tqdm import tqdm_notebook as tqdm

import requests

import os

import json

import pandas as pd

import csv

import datetime
import dateutil.parser
import unicodedata

import time
import requests

### Part A — General Exploration

How many games does RAWG have registered in total?

Print the number with a clear message.

El procedimiento es, en primer lugar jalar la url donde esta la sección juegos con nustra API key para luego descargarlo en un formato JSON. De ahi vemos las keys de jugos para ver que existe una con el nombre count que alberga el conteo de todos los juegos registrados por RAWG. Finalmente con un print mostramos la data de cuantos juegos han sido contados.


In [ ]:
url = f"https://api.rawg.io/api/games?key={api_key}"
info = requests.get(url)

In [ ]:
juegos = info.json()
juegos


{'count': 898364,
 'next': 'https://api.rawg.io/api/games?key=74c8e14f7976452ebdbdc1fcf6b714c9&page=2',
 'previous': None,
 'results': [{'id': 3498,
   'slug': 'grand-theft-auto-v',
   'name': 'Grand Theft Auto V',
   'released': '2013-09-17',
   'tba': False,
   'background_image': 'https://media.rawg.io/media/games/20a/20aa03a10cda45239fe22d035c0ebe64.jpg',
   'rating': 4.47,
   'rating_top': 5,
   'ratings': [{'id': 5,
     'title': 'exceptional',
     'count': 4422,
     'percent': 58.95},
    {'id': 4, 'title': 'recommended', 'count': 2462, 'percent': 32.82},
    {'id': 3, 'title': 'meh', 'count': 471, 'percent': 6.28},
    {'id': 1, 'title': 'skip', 'count': 146, 'percent': 1.95}],
   'ratings_count': 7365,
   'reviews_text_count': 74,
   'added': 22549,
   'added_by_status': {'yet': 560,
    'owned': 12877,
    'beaten': 6489,
    'toplay': 652,
    'dropped': 1192,
    'playing': 779},
   'metacritic': 92,
   'playtime': 74,
   'suggestions_count': 450,
   'updated': '2026-04-0

In [27]:
juegos.keys()

dict_keys(['count', 'next', 'previous', 'results', 'seo_title', 'seo_description', 'seo_keywords', 'seo_h1', 'noindex', 'nofollow', 'description', 'filters', 'nofollow_collections'])

In [30]:
count = juegos['count']
print("El número de juegos que RAWG tiene registrados en total es", count) 

El número de juegos que RAWG tiene registrados en total es 898364


### Part B — Category Analysis

B1

What is the top 5 highest rated games of all time according to Metacritic?

Show: name, rating, and metacritic score.

En primer lugar jalamos el mismo url que en la primera pero con un pequeño artificio, ordenamos la informacion de forma descendente por metacritic dado que nos dicen que busquemos los 5 mejores juegos rateados SEGUN metacritic. Creamos una tabla donde extraemos para los primeros 5 ordenados de mayor a menos es decir el top 5 su name, rating, metacritic para luego exportarlo a un dataframe.

In [143]:
url1 = f"https://api.rawg.io/api/games?key={api_key}&ordering=-metacritic"
info1 = requests.get(url1)

In [210]:
critic = info1.json()

table = []

for i in range(5):
    name = critic['results'][i]['name']
    rating = critic['results'][i]['rating']
    metacritic = critic['results'][i]['metacritic']

    fila = name, rating, metacritic
    table.append(fila)

df = pd.DataFrame(table, columns=["name", "rating", "metacritic"])
df


,name,rating,metacritic
0,The Legend of Zelda: Ocarina of Time,4.38,99
1,Soulcalibur (1998),0.00,98
2,Soulcalibur,4.38,98
3,Baldur's Gate III,4.44,97
4,Metroid Prime,4.35,97


B2

What are the 10 best games available on Steam (store_id=1)?

Show name, rating, and metacritic score.

Es el mismo url que en la anterior pregunta solo que ahora tambien filtramos por stores donde como se nos dio, el store_id = 1 para la tienda de Steam por lo que extraemos de esa url su formato JSON para luego en una tabla extraer el name, raitng and metacritic de los 10 primeros que dado que estan ordenados de forma descendente por metacritic nos daran los 10 mejores juegos de steam ordenados por metacritic. Finalmente lo exportamos a un dataframe.

In [208]:
url2 = f"https://api.rawg.io/api/games?key={api_key}&stores=1&ordering=-metacritic"
info2 = requests.get(url2)

In [211]:
steam_rating = info2.json()

table1 = []

for i in range(10):
    name = steam_rating['results'][i]['name']
    rating = steam_rating['results'][i]['rating']
    metacritic = steam_rating['results'][i]['metacritic']

    fila = name, rating, metacritic
    table1.append(fila)

df1 = pd.DataFrame(table1, columns=["name", "rating", "metacritic"])
df1


,name,rating,metacritic
0,Baldur's Gate III,4.44,97
1,Half-Life 2: Update,4.13,96
2,Half-Life,4.38,96
3,Red Dead Redemption 2,4.59,96
4,Half-Life 2,4.48,96
5,BioShock,4.36,96
6,Grand Theft Auto IV: Complete Edition,4.57,95
7,Divinity: Original Sin 2,4.38,95
8,Portal 2,4.58,95
9,Red Dead Redemption,4.42,95


### Part C — Comparisons  

C1

Compare the top 5 games on PC (platform_id=4) vs top 5 on PS5 (platform_id=187).

Which platform has the highest rated games?

Extraemos las urls para las plataformas de PC (platform_id=4) y PS5 (platform_id=187) y realizamos el mismo proceso de antes para un top 5 mejores juegos de cada plataforma. Finalmente comparamos las columnas de rating de PC vs PS5 lo que nos da como resultado que los juegos de PC tienen mejor rating.

In [215]:
url3 = f"https://api.rawg.io/api/games?key={api_key}&platforms=4&ordering=-rating"
info3 = requests.get(url3)

In [218]:
pc_games = info3.json()
table2 = []

for i in range(5):
    name = pc_games['results'][i]['name']
    rating = pc_games['results'][i]['rating']
    metacritic = pc_games['results'][i]['metacritic']

    fila = name, rating, metacritic
    table2.append(fila)

df2 = pd.DataFrame(table2, columns=["name", "rating", "metacritic"])
df2

,name,rating,metacritic
0,The Elder Scrolls VI,4.86,NaN
1,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN
2,No Case Should Remain Unsolved,4.83,NaN
3,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0
4,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0


In [219]:
url4 = f"https://api.rawg.io/api/games?key={api_key}&platforms=187&ordering=-rating"
info4 = requests.get(url4)

In [221]:
ps5_games = info4.json()
table3 = []

for i in range(5):
    name = ps5_games['results'][i]['name']
    rating = ps5_games['results'][i]['rating']
    metacritic = ps5_games['results'][i]['metacritic']

    fila = name, rating, metacritic
    table3.append(fila)

df3 = pd.DataFrame(table3, columns=["name", "rating", "metacritic"])
df3

,name,rating,metacritic
0,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0
1,Persona 5 Royal,4.75,94.0
2,Cyberpunk 2077: Phantom Liberty,4.71,NaN
3,The Binding of Isaac: Repentance,4.69,NaN
4,The Last of Us Part I,4.67,NaN


In [225]:
df3['rating'] > df2['rating']

0    False
1    False
2    False
3    False
4    False
Name: rating, dtype: bool

C2

Choose 3 famous games and build a comparison table with:

name, rating, metacritic, genres, and platforms.

Ahora hacemos un request de los mejores juegos ordenados por rating y seleccionamos los juegos de nuestra preferencia (3), despues extraemos de platforms las diferentes plataformas donde se pueden jugar dichos juegos para cada uno de los 3 juegos seleccionados. Finalmente exportamos la data a un dataframe donde esta el name, rating, metacritic and platforms que es lo que agregamos anteriormente.

In [226]:
url5 = f"https://api.rawg.io/api/games?key={api_key}&ordering=-rating"
info5 = requests.get(url5)

In [244]:
best_rated_games = info5.json()
table4 = []

for i in range(20):
    name = best_rated_games['results'][i]['name']
    rating = best_rated_games['results'][i]['rating']
    metacritic = best_rated_games['results'][i]['metacritic']

    fila = name, rating, metacritic
    table4.append(fila)

df4 = pd.DataFrame(table4, columns=["name", "rating", "metacritic"])
df4



,name,rating,metacritic
0,The Elder Scrolls VI,4.86,NaN
1,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN
2,No Case Should Remain Unsolved,4.83,NaN
3,Gimmick!,4.83,NaN
4,Super Robot Taisen: Original Generation,4.83,NaN
5,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0
6,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0
7,The Witcher 3: Wild Hunt – Hearts of Stone,4.76,90.0
8,Persona 5 Royal,4.75,94.0
9,Mass Effect Trilogy,4.75,NaN


In [291]:
table5 = []
for x in [6, 8, 11]:
    name = best_rated_games['results'][x]['name']
    rating = best_rated_games['results'][x]['rating']
    metacritic = best_rated_games['results'][x]['metacritic']
    platforms = []
    i = 0
    while True:
        try:
            platform = best_rated_games['results'][x]['platforms'][i]['platform']['name']
            platforms.append(platform)
            i = i + 1
        except IndexError:
            print("Se acabaron los elementos")
            break
    fila = name, rating, metacritic, platforms
    table5.append(fila)

df5 = pd.DataFrame(table5, columns=["name", "rating", "metacritic", "platforms"])
df5


Se acabaron los elementos
Se acabaron los elementos
Se acabaron los elementos


,name,rating,metacritic,platforms
0,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0,"[Xbox One, Xbox Series S/X, PC, Nintendo Switc..."
1,Persona 5 Royal,4.75,94.0,"[Xbox Series S/X, PC, PlayStation 5, Nintendo ..."
2,Cyberpunk 2077: Phantom Liberty,4.71,NaN,"[PlayStation 5, Xbox Series S/X, PC]"


C3

Query the top 5 games from at least 4 different genres, calculate the average rating for each, and determine which genre produces the best games according to users.

En primer lugar sacamos los json files de los generos con ids del 1 al 5. Luego mediante appends hacemos un dataframe con columnas donde cada columna representa los ratings de los top 5 mejores juegos de cada genero. A continuacion se halla por columnas el promedio general de cada uno de los generos en cuanto a rating para ver que el que tiene mejor rating es el genre_5 por lo que buscamos en el json_5 que alberga la data del genero 5 para indagar que el genre_5 hace referencia a los juegos RPG.


In [317]:
data_json = {}

for i in [1,2,3,4,5]:
    genero = f"https://api.rawg.io/api/games?key={api_key}&genres={i}&ordering=-rating"
    informacion = requests.get(genero)

    data_json[f"json_{i}"] = informacion.json()

In [355]:
ratings = pd.DataFrame()
for x in [1,2,3,4,5]:
    tabla = []
    
    for i in range(5):
        rating = data_json[f"json_{x}"]['results'][i]['rating']
        tabla.append(rating)
    
    ratings[f"genre_{x}"] = tabla

ratings


,genre_1,genre_2,genre_3,genre_4,genre_5
0,4.67,4.75,4.83,4.86,4.86
1,4.67,4.71,4.80,4.83,4.81
2,4.62,4.67,4.75,4.75,4.80
3,4.58,4.67,4.71,4.71,4.76
4,4.58,4.62,4.71,4.71,4.75


In [382]:
fila = {}

for x in [1,2,3,4,5]:
    fila[f"genre_{x}"] = ratings[f"genre_{x}"].mean()

rating_mean = pd.DataFrame([fila])
rating_mean

,genre_1,genre_2,genre_3,genre_4,genre_5
0,4.624,4.684,4.76,4.772,4.796


In [384]:
mejor_genero = rating_mean.idxmax(axis=1)[0]
mejor_genero

'genre_5'

In [ ]:
data_json['json_5']['results'][1]['genres'][0]['name']

'RPG'

C4 

Compare the best games from 3 different years of your choice.

In which year were the games with the highest average metacritic score released?

Seleccionamos los datos de los años 2012, 2018 y 2020. Los ordenamos de manera descendente en metacritica para luego extraer los top 20 mejores juegos y sus respectivas metacriticas. Finalmente siguiendo pasos similares al anterior ejercicio, extraemos las columnas donde cada columna representa un año y sus filas los top 20 mejores reseñas de metacriticas. Finalmente hallamos un promedio y obtenemos que el año con mejores metacriticas en promedio es el 2020.

In [409]:
year_2012 = f"https://api.rawg.io/api/games?key={api_key}&dates=2012-01-01,2012-12-31&ordering=-metacritic"
year_2018 = f"https://api.rawg.io/api/games?key={api_key}&dates=2018-01-01,2018-12-31&ordering=-metacritic"
year_2020 = f"https://api.rawg.io/api/games?key={api_key}&dates=2020-01-01,2020-12-31&ordering=-metacritic"
info_2012 = requests.get(year_2012)
info_2018 = requests.get(year_2018)
info_2020 = requests.get(year_2020)

In [422]:
juegos_2012 = info_2012.json()
juegos_2018 = info_2018.json()
juegos_2020 = info_2020.json()

juegos = {
    2012: juegos_2012,
    2018: juegos_2018,
    2020: juegos_2020
}

In [430]:
metacritics = pd.DataFrame()
for x in [2012, 2018, 2020]:
    tabla = []

    for i in range(20):
        metacritic = juegos[x]['results'][i]['metacritic']
        tabla.append(metacritic)
    
    metacritics[x] = tabla

metacritics

,2012,2018,2020
0,92,96,94
1,92,94,93
2,92,94,93
3,91,93,93
4,91,91,92
5,91,91,92
6,91,90,91
7,90,90,90
8,90,90,90
9,90,90,90


In [431]:
fila = {}

for x in [2012, 2018, 2020]:
    fila[x] = metacritics[x].mean()

metacritics_mean = pd.DataFrame([fila])
metacritics_mean

,2012,2018,2020
0,89.7,90.0,90.15


In [435]:
mejor_año = metacritics_mean.idxmax(axis=1)[0]
print(mejor_año)

2020


C5

Export the top 20 games of all time to a CSV file named top20_rawg.csv inside the api/output/ folder.

Extraemos los datos de los 20 mejores juegos de todos los tiempos, similar al ejercicio 1 sin embargo esta vez seran 20 en lugar de 5 y añadiremos mas columnas de informacion como el released_date y main_genre. Creamos un dataframe y luego exportamos la data a un csv. Finalmente con el comando .head() se muestran los 5 primeros juegos con las columnas name, rating, metacritic, released_date y main_genre.

In [437]:
final_url = f"https://api.rawg.io/api/games?key={api_key}&ordering=-metacritic"
info_final = requests.get(final_url)

In [453]:
final_json = info_final.json()
table_final = []

for i in range(20):
    name = final_json['results'][i]['name']
    rating = final_json['results'][i]['rating']
    metacritic = final_json['results'][i]['metacritic']
    release_date = final_json['results'][i]['released']
    main_genre = final_json['results'][i]['genres'][0]['name']

    fila = name, rating, metacritic, release_date, main_genre
    table_final.append(fila)

df_final = pd.DataFrame(table_final, columns=["name", "rating", "metacritic", "release_date", "main_genre"])
df_final

,name,rating,metacritic,release_date,main_genre
0,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
1,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
2,Soulcalibur,4.38,98,1998-07-30,Action
3,Baldur's Gate III,4.44,97,2023-08-03,Adventure
4,Metroid Prime,4.35,97,2002-11-17,Action
5,Perfect Dark,3.98,97,2000-05-22,Action
6,Super Mario Odyssey,4.42,97,2017-10-27,Arcade
7,Super Mario Galaxy 2,4.34,97,2010-05-23,Platformer
8,Super Mario Galaxy,4.35,97,2007-11-01,Platformer
9,The Legend of Zelda: Breath of the Wild,4.47,97,2017-03-03,Action


In [455]:
df_final.to_csv(r'..\API\Output\top20_rawg.csv', index=False)

In [456]:
df_final.head()

,name,rating,metacritic,release_date,main_genre
0,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
1,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
2,Soulcalibur,4.38,98,1998-07-30,Action
3,Baldur's Gate III,4.44,97,2023-08-03,Adventure
4,Metroid Prime,4.35,97,2002-11-17,Action
